In [11]:
from numba import njit, int64, float64
from numba.experimental import jitclass
from numba.typed import List
from numba import types
import numpy as np


# PAIR_TYPE = types.Tuple((float64, int64))
TUPLE_TYPE = types.Tuple((float64, int64))

heap_spec = [
    ('data', types.ListType(TUPLE_TYPE))
]

@jitclass(heap_spec)
class Heap:

    def __init__(self):
        self.data = List.empty_list(TUPLE_TYPE)

    def push(self, key, idx):
        self.data.append((key, idx))
        i = len(self.data) - 1
        while i > 0:
            parent = (i - 1) // 2
            if self.data[i][0] <= self.data[parent][0]:
                break
            self.data[i], self.data[parent] = self.data[parent], self.data[i]
            i = parent

    def pop(self):
        top = self.data[0]
        last = self.data.pop()
        if len(self.data) == 0:
            return top
        self.data[0] = last
        i = 0
        while True:
            left = 2 * i + 1
            right = 2 * i + 2
            largest = i
            if left < len(self.data) and self.data[left][0] > self.data[largest][0]:
                largest = left
            if right < len(self.data) and self.data[right][0] > self.data[largest][0]:
                largest = right
            if largest == i:
                break
            self.data[i], self.data[largest] = self.data[largest], self.data[i]
            i = largest
        return top
    
heap = Heap()
heap.push(-1.0, 0)
heap.pop()

(-1.0, 0)

In [ ]:
import numpy as np

from numba.experimental import jitclass
from numba.types import int64, float64, FunctionType
from numba.typed import List
from numba import njit

from opticon import Propositionalization, compute_bounds, equal_width_propositionalization
from testdata import mvn_with_correlation

@jitclass
class CanonicalTreeSearchNode:

    key: int64[:]
    critical: int64[:]
    remaining: int64[:]
    support: int64[:]

    def __init__(self, key, critical, remaining, support):
        self.key = key
        self.critical = critical
        self.remaining =remaining
        self.support = support

def node_to_string(node):
    return f'Node({node.key}, {node.value}, {node.bound})'

@njit
def dummy_obj(node):
    return float64(len(node.key))

@njit
def dummy_bnd(node):
    return float64(len(node.key) + len(node.remaining))

CanonicalTreeSearchNodeType = CanonicalTreeSearchNode.class_type.instance_type

@jitclass
class CanonicalTreeSearch:

    x: float64[:, :]
    prop: Propositionalization

    def __init__(self, x, prop):
        self.x = x
        self.prop = prop

    def refinement(self, node):
        res = List()
        for p_idx in range(len(node.remaining)):
            p = node.remaining[p_idx]
            _key = np.append(node.key, p)
            _sup = self.prop.support(p, self.x[node.support])
            _crit = np.append(node.critical, node.remaining[:p_idx])
            l, u = compute_bounds(self.x[_sup])
            if len(self.prop.trivial(l, u, _crit))>0: #canonicity check
                continue
            _remaining = self.prop.nontrivial(l, u, node.remaining[p_idx+1:])
            res.append(CanonicalTreeSearchNode(_key, _crit, _remaining, _sup))
        return res

    def make_root(self):
        l, u = compute_bounds(self.x)
        remaining = self.prop.nontrivial(l, u, np.arange(len(self.prop)))
        empty = np.empty(0, dtype=np.int64)
        return CanonicalTreeSearchNode(empty, empty, remaining, np.arange(len(self.x)))
    
    def run(self, max_depth = 10):
        heap = Heap()
        nodes = List.empty_list(CanonicalTreeSearchNodeType) #List.empty_list(CanonicalTreeSearchNode.class_type.instance_type)
        freelist = List.empty_list(int64)

        root = self.make_root()
        nodes.append(root)
        heap.push(-dummy_bnd(root), 0)

        best_value = dummy_obj(root)
        created = 1

        while len(heap.data) > 0:
            neg_bound, idx = heap.pop()
            node = nodes[idx]
            freelist.append(idx)

            if -neg_bound < best_value:
                continue
            if len(node.key) >= max_depth:
                continue

            children = self.refinement(node)
            created += len(children)

            for child in children:
                val = dummy_obj(child)
                bnd = dummy_bnd(child)
                if val > best_value:
                    best_value = val

                if len(freelist) > 0:
                    reuse_idx = freelist.pop()
                    nodes[reuse_idx] = child
                    heap.push(-bnd, reuse_idx)
                else:
                    nodes.append(child)
                    heap.push(-bnd, len(nodes) - 1)

        print("Best value:", best_value)
        print("Total nodes created:", created)
        return best_value
        

x = mvn_with_correlation(100)
search = CanonicalTreeSearch(x, equal_width_propositionalization(x))
search.run()

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
- Resolution failure for literal arguments:
Failed in nopython mode pipeline (step: nopython frontend)
Unknown attribute 'class_type' of type jitclass.CanonicalTreeSearchNode#11bd75fd0<key:array(int64, 1d, A),critical:array(int64, 1d, A),remaining:array(int64, 1d, A),support:array(int64, 1d, A)>

File "../../../../../var/folders/dw/3pgtt23n4gzd5kwldjcrppq80000gn/T/ipykernel_21880/294119894.py", line 68:
<source missing, REPL/exec in use?>

During: typing of get attribute at /var/folders/dw/3pgtt23n4gzd5kwldjcrppq80000gn/T/ipykernel_21880/294119894.py (68)

File "../../../../../var/folders/dw/3pgtt23n4gzd5kwldjcrppq80000gn/T/ipykernel_21880/294119894.py", line 68:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
- Resolution failure for non-literal arguments:
None

During: resolving callee type: BoundFunction((<class 'numba.core.types.misc.ClassInstanceType'>, 'run') for instance.jitclass.Ca